# Fine-tuning hospitalar acadêmico no Google Colab
Selecione **Ambiente de execução > Alterar tipo > GPU T4**. Execute em ordem. O treinamento real é necessário para concluir o desafio. Sem dados reais. O repositório pode ser privado: use o formulário seguro abaixo. Nunca salve tokens no notebook.

In [ ]:
import os, subprocess, getpass
from pathlib import Path
REPO = "https://github.com/felipepereira598-cmd/techChallenge3.git"
PRIVATE_REPO = False
if not Path("/content/medical-assistant").exists():
    if PRIVATE_REPO:
        token = getpass.getpass("Token GitHub com leitura deste repositório (não será salvo): ")
        import base64
        env = os.environ.copy()
        env.update(GIT_CONFIG_COUNT="1", GIT_CONFIG_KEY_0="http.https://github.com/.extraheader", GIT_CONFIG_VALUE_0="AUTHORIZATION: basic " + base64.b64encode(("x-access-token:"+token).encode()).decode())
        subprocess.run(["git","clone",REPO,"/content/medical-assistant"],env=env,check=True)
        del token, env
    else:
        subprocess.run(["git","clone",REPO,"/content/medical-assistant"],check=True)
os.chdir("/content/medical-assistant")
print(subprocess.check_output(["git","rev-parse","HEAD"],text=True))

In [ ]:
%pip install -q -r requirements-training.txt

Se o Colab pedir reinicialização, reinicie e execute novamente a primeira célula (o clone será reaproveitado).

In [ ]:
import torch, subprocess
from pathlib import Path
assert torch.cuda.is_available(), "Ative a GPU antes de continuar"
print(torch.cuda.get_device_name(0))
for url, target in [("https://github.com/abachaa/MedQuAD.git", "data/raw/MedQuAD"), ("https://github.com/pubmedqa/pubmedqa.git", "data/raw/pubmedqa")]:
    if not Path(target).exists():
        subprocess.run(["git", "clone", url, target], check=True)
subprocess.run(["python", "-m", "scripts.prepare_medquad", "--raw", "data/raw/MedQuAD", "--limit", "3000"], check=True)
subprocess.run(["python", "-m", "scripts.prepare_pubmedqa", "--input", "data/raw/pubmedqa/data/ori_pqal.json"], check=True)

## Curadoria humana antes do treinamento
Inspecione a amostra e os descartes. Verifique fontes, ausência de identificadores e adequação ao escopo. Os documentos hospitalares são fictícios. Registre revisões no relatório.

In [ ]:
import json
from pathlib import Path
print(Path("data/processed/manifest.json").read_text())
for line in Path("data/processed/medquad_train.jsonl").read_text().splitlines()[:3]:
    print(json.loads(line))

In [ ]:
from huggingface_hub import model_info
MODEL = "Qwen/Qwen2.5-1.5B-Instruct"
REVISION = model_info(MODEL).sha
print("Revisão fixada:", REVISION)
import subprocess
subprocess.run(["python","-m","scripts.train","--qlora","--model",MODEL,"--revision",REVISION],check=True)

In [ ]:
subprocess.run(["python","-m","scripts.evaluate","--quantized","--model",MODEL,"--revision",REVISION,"--limit","50"],check=True)
print(Path("results/comparison.json").read_text())

## Exportar evidências
O ZIP inclui adapter, manifestos, métricas e versões. Revise as respostas com a rubrica humana. Não conclua superioridade clínica a partir de token F1 ou de 50 exemplos. Baixe e extraia models/adapter na raiz do projeto local. Reinicie o Streamlit após substituir o adapter.

In [ ]:
import shutil
subprocess.run("python -m pip freeze > results/environment.txt",shell=True,check=True)
Path("results/data_manifest.json").write_text(Path("data/processed/manifest.json").read_text())
Path("export").mkdir(exist_ok=True)
shutil.copytree("models", "export/models", dirs_exist_ok=True)
shutil.copytree("results", "export/results", dirs_exist_ok=True)
shutil.make_archive("training-artifacts","zip","export")
from google.colab import files
files.download("training-artifacts.zip")